# 06. Comparison of all models

Verdict: LightGBM residual (final model of 05) wins, tied with the other LightGBM using RTE on average but better on cold-day evening peaks. MAE test: 910 MW vs RTE raw 1,439 MW (-36.8 %).

Setup: Cutoff D 12:00, forecast the 24 h of D+1, test 1 Jul 2025 to 30 Jun 2026.

Nothing is fitted here, we load the test forecasts saved by 01 to 05 (run them first).

## 1. Setup

In [1]:
import sys
sys.path.insert(0, "../src")

import polars as pl
from watt_forecast.data_preprocessing import DAILY_KEYS, HOURLY_KEYS, PREDICTIONS_DIR, load_predictions
from watt_forecast.metrics import scorecard, scorecard_daily

HOURLY = ["01_baseline", "04_lightgbm", "05_final_model"]
DAILY = ["02_ar", "03_sarima"]

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)


def sources(notebooks, keys):
    """{forecast name: notebook} for the forecasts saved by `notebooks`."""
    return {
        name: nb for nb in notebooks for name in pl.read_csv(PREDICTIONS_DIR / f"{nb}.csv", n_rows=0).columns if name not in keys
    }


hourly, hourly_src = load_predictions(HOURLY), sources(HOURLY, HOURLY_KEYS)
daily, daily_src = load_predictions(DAILY, keys=DAILY_KEYS), sources(DAILY, DAILY_KEYS)
assert hourly.height == 365 * 24, "the hourly test period should have 8,760 hours"
assert daily.height == 365, "the daily test period should have 365 days"
assert hourly["datetime_fr"].is_unique().all(), "each local datetime (with its offset) should appear once"
assert len(hourly_src) == hourly.drop(*HOURLY_KEYS, "date", "hour", "temperature_c").width, "a forecast name is saved by two notebooks"
assert len(daily_src) == len(daily.columns) - len(DAILY_KEYS), "a forecast name is saved by two notebooks"

## 2. Hourly models (MW)

In [2]:
scorecard(hourly, {name: name for name in hourly_src}, reference="RTE_raw").select(
    pl.col("forecast").replace_strict(hourly_src).alias("notebook"), pl.all()
).sort("global")

notebook,forecast,global,evening_peak_18-20h,evening_peak_cold_days,vs_reference_(%)
str,str,f64,f64,f64,f64
"""04_lightgbm""","""LightGBM_+_RTE""",908.0,984.0,2010.0,-36.9
"""05_final_model""","""LightGBM_residual""",910.0,987.0,1906.0,-36.8
"""04_lightgbm""","""LightGBM_observed_temp._(UPPER_BOUND,_not_available)""",1184.0,1220.0,2170.0,-17.7
"""01_baseline""","""RTE_raw""",1439.0,1348.0,2307.0,0.0
"""04_lightgbm""","""LightGBM_persistence""",2010.0,2036.0,4466.0,39.7
"""04_lightgbm""","""Linear_regression""",2299.0,2279.0,4867.0,59.8
"""01_baseline""","""Lag_7d""",3425.0,3612.0,9225.0,138.0
"""01_baseline""","""Lag_2d""",3857.0,4026.0,4346.0,168.0


Only models using RTE's forecast (or the observed temperature) beat RTE raw. The two LightGBM + RTE models are tied (908 and 910 MW), the residual one is better on cold-day evening peaks (1,906 vs 2,010 MW). Our best honest model without RTE is still 39.7 % worse than RTE raw.

## 3. Daily models (GW)

Daily mean load (milestones m2/m3), not comparable with the hourly table.

In [3]:
scorecard_daily(daily, {name: name for name in daily_src}, reference="RTE_raw").select(
    pl.col("forecast").replace_strict(daily_src).alias("notebook"), pl.all()
).sort("MAE")

notebook,forecast,MAE,bias,vs_reference_(%)
str,str,f64,f64,f64
"""02_ar""","""RTE_raw""",1.16,-1.09,0.0
"""03_sarima""","""SARIMAX_observed_temp._(upper_bound)""",1.16,-0.01,0.2
"""03_sarima""","""SARIMAX_persistence""",1.65,-0.0,42.5
"""02_ar""","""AR(8)_+_normal""",1.69,-0.17,46.3
"""03_sarima""","""SARIMA""",1.79,-0.01,54.7
"""03_sarima""","""Holt-Winters""",1.83,0.01,58.1
"""02_ar""","""Normal_only""",2.66,-0.6,129.9
"""02_ar""","""Seasonal_naive_(T-7)""",3.29,-0.05,185.1


No daily model beats RTE. Best honest one is SARIMAX with T-2 temperature (1.65 GW), just ahead of the AR (1.69 GW).

## 4. Conclusion

- Deploy LightGBM residual (05), RTE's forecast plus a learnt correction.
- Without RTE our models never catch up. The value is in correcting RTE, not replacing it.
- Kept for next step: the deployment checks from 05 (RTE published before 12:00, real-time data, retraining, wider intervals on cold days).